In [ ]:
# 프로젝트 위치가 GummyBearing인지 확인

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("프로젝트의 src 폴더를 찾을 수 없습니다.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.xjtu_features import extract_xjtu_features

print("프로젝트 위치:", PROJECT_ROOT)

In [ ]:
# 함수를 불러오고 데이터 경로와 베어링 폴더 확인

import numpy as np
import pandas as pd
from IPython.display import display

from src.xjtu_features import (
    extract_xjtu_features, find_bearing_folders,
    BEARING_METADATA, FEATURE_COLUMNS, VALUE_COLUMNS, ERROR_COLUMNS,
)

# 압축을 푼 세 조건의 폴더가 들어 있는 상위 경로로 수정한다.
DATA_ROOT = PROJECT_ROOT / "DATA" / "XJTU_SY_Bearing_Datasets"

# 기본은 15개 전체. 일부만 시험하려면 ["Bearing1_1"] 등으로 바꾼다.
selected_bearings = list(BEARING_METADATA)
bearing_folders = find_bearing_folders(DATA_ROOT, selected_bearings)

for bearing, folder in bearing_folders.items():
    count = sum(p.is_file() and p.suffix.lower() == ".csv" for p in folder.iterdir())
    expected = BEARING_METADATA[bearing]["expected_files"]
    print(f"{bearing}: CSV {count}개 / PDF 기준 {expected}개 | {folder}")

# timestamp: 파일명 숫자(1, 2, ...), 실제 날짜가 아닌 1분 간격 측정 순서.
# fault_element: 종료 후 확인된 결함 부위. 개별 시점의 정상/고장 라벨이 아니다.
# 메타데이터 출처: Introduction_to_XJTU-SY_Bearing_Dataset, p.3 Table 2.

In [ ]:
# 특징 계산 실행 (원본 CSV 하나 → 수평·수직 특징을 담은 한 행)

results = {}
error_results = {}

for bearing, folder in bearing_folders.items():
    features_df, errors_df = extract_xjtu_features(
        folder_path=folder,
        bearing=bearing,
    )
    results[bearing] = features_df
    error_results[bearing] = errors_df
    print(f"{bearing}: 결과 {features_df.shape}, 오류 항목 {len(errors_df)}개\n")

In [ ]:
# 결과 확인

summary_df = pd.DataFrame([
    {
        "bearing": bearing,
        "condition": BEARING_METADATA[bearing]["condition"],
        "expected_rows": BEARING_METADATA[bearing]["expected_files"],
        "result_rows": len(features_df),
        "error_count": len(error_results[bearing]),
    }
    for bearing, features_df in results.items()
])
display(summary_df)

xjtu_df = pd.concat(results.values(), ignore_index=True)
xjtu_df = xjtu_df.sort_values(["condition", "bearing", "timestamp"]).reset_index(drop=True)

# 14열을 생략하지 않고 미리 본다. 저장할 때는 반올림하지 않는다.
with pd.option_context("display.max_columns", None, "display.precision", 4):
    display(xjtu_df.head())
print("통합 결과 크기:", xjtu_df.shape)

In [ ]:
# 오류와 계산된 특징 확인 (원본 결측치 일괄 검사는 팀원 담당)

all_errors_df = pd.concat(error_results.values(), ignore_index=True).reindex(columns=ERROR_COLUMNS)
print("오류 항목 수:", len(all_errors_df))
if not all_errors_df.empty:
    display(all_errors_df)

print("계산된 특징의 NaN 개수:")
display(xjtu_df[VALUE_COLUMNS].isna().sum())

# 다른 베어링에서 timestamp가 같은 것은 정상이다.
duplicate_mask = xjtu_df.duplicated(["bearing", "timestamp"], keep=False)
print("중복된 bearing + timestamp 행 수:", int(duplicate_mask.sum()))
if duplicate_mask.any():
    display(xjtu_df.loc[duplicate_mask])

print("숫자 정렬 및 파일 번호 누락·중복 확인은 추출 함수에서 수행합니다.")
print("전처리에서는 보간·이상치 삭제·정규화·이동평균을 적용하지 않습니다.")

In [ ]:
# 베어링별 CSV와 통합 CSV 저장

OUTPUT_ROOT = PROJECT_ROOT / "DATA" / "features"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ERROR_ROOT = OUTPUT_ROOT / "errors"
ERROR_ROOT.mkdir(parents=True, exist_ok=True)

# 저장 직전 현재 결과에서 통합본과 오류 목록을 다시 구성한다.
xjtu_df = pd.concat(results.values(), ignore_index=True).reindex(columns=FEATURE_COLUMNS)
xjtu_df = xjtu_df.sort_values(["condition", "bearing", "timestamp"]).reset_index(drop=True)
all_errors_df = pd.concat(error_results.values(), ignore_index=True).reindex(columns=ERROR_COLUMNS)
all_errors_df.to_csv(ERROR_ROOT / "XJTU_SY_errors.csv", index=False, encoding="utf-8-sig")

has_invalid_features = not np.isfinite(xjtu_df[VALUE_COLUMNS].to_numpy(dtype=float)).all()
has_duplicates = xjtu_df.duplicated(["bearing", "timestamp"]).any()
has_wrong_count = any(
    len(frame) != BEARING_METADATA[bearing]["expected_files"]
    for bearing, frame in results.items()
)
if xjtu_df.empty or not all_errors_df.empty or has_invalid_features or has_duplicates or has_wrong_count:
    raise ValueError("오류/특징값/파일 개수를 확인하세요. 오류 목록만 저장했고 특징 CSV는 갱신하지 않았습니다.")

for bearing, features_df in results.items():
    # Bearing1_1 → XJTU_SY_bearing_1_1_features.csv
    bearing_id = bearing.removeprefix("Bearing")
    filename = f"XJTU_SY_bearing_{bearing_id}_features.csv"
    features_df.to_csv(OUTPUT_ROOT / filename, index=False, encoding="utf-8-sig")

# 일부 베어링으로 시험한 결과를 전체 통합본으로 저장하지 않는다.
is_full_dataset = set(results) == set(BEARING_METADATA)
total_name = "XJTU_SY_total_features.csv" if is_full_dataset else "XJTU_SY_selected_features.csv"
xjtu_df.to_csv(OUTPUT_ROOT / total_name, index=False, encoding="utf-8-sig")
print(f"베어링별 {len(results)}개 + 통합 1개 저장 완료:", OUTPUT_ROOT)
print("통합 파일:", total_name, "| 크기:", xjtu_df.shape)